# ProcureCompare AI

**Intelligent Product Sourcing and Price Comparison Agent**

In [ ]:
# Run this cell once after selecting the Python 3.12 (ai-agent) kernel.
%pip install -q -r requirements.txt

In [ ]:
import sys

# Fail early when the selected notebook kernel uses an unsupported Python version.
print(f"Python: {sys.version.split()[0]}")
print(f"Interpreter: {sys.executable}")
assert sys.version_info[:2] in {(3, 10), (3, 11), (3, 12)}, (
    "This project requires Python 3.10-3.12. Select the Python 3.12 (ai-agent) kernel."
)

In [ ]:
# CrewAI provides the agents, tasks, tools, and sequential workflow orchestration.
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool
from crewai.knowledge.source.string_knowledge_source import StringKnowledgeSource
import agentops
from agentops.sdk.exporters import AuthenticatedOTLPExporter
from agentops.sdk import processors as agentops_processors
from pydantic import BaseModel, Field
from typing import List, Optional
from tavily import TavilyClient
from scrapegraph_py import ScrapeGraphAI, FetchConfig
from dotenv import load_dotenv
from pathlib import Path
from types import SimpleNamespace

import os
import json

# Support kernels started either from this project folder or from the repository root.
env_candidates = [Path.cwd() / ".env", Path.cwd() / "agents/procurecompare-ai/.env"]
env_path = next((path for path in env_candidates if path.is_file()), None)
if env_path is None:
    raise FileNotFoundError("Local .env file not found. Copy .env.example to .env first.")
project_dir = env_path.parent
repository_root = project_dir.parents[1] if project_dir.parent.name == "agents" else project_dir

# Replace stale values inherited by the VS Code/Jupyter server.
load_dotenv(dotenv_path=env_path, override=True)

def get_secret(
    env_name: str,
    *,
    required: bool = True,
) -> Optional[str]:
    """Read a secret from the macOS process environment or local .env file."""
    value = os.getenv(env_name)
    if value and not value.startswith("your_"):
        return value
    if not required:
        return None

    raise RuntimeError(
        f"Missing {env_name}. Add it to the local .env file, then rerun this cell."
    )

In [ ]:
# OpenAI is required by every CrewAI agent; AgentOps tracing is optional.
openai_api_key = get_secret("OPENAI_API_KEY")
if not openai_api_key.startswith("sk-"):
    raise RuntimeError("OPENAI_API_KEY has an invalid format. Use an OpenAI secret key beginning with 'sk-'.")
agentops_api_key = get_secret("AGENTOPS_API_KEY", required=False)
os.environ["OPENAI_API_KEY"] = openai_api_key

# Compatibility for AgentOps 0.4.21 with OpenTelemetry 1.45.
if not hasattr(AuthenticatedOTLPExporter, "_procurecompare_original_init"):
    AuthenticatedOTLPExporter._procurecompare_original_init = AuthenticatedOTLPExporter.__init__

def _compatible_exporter_init(self, *args, **kwargs):
    AuthenticatedOTLPExporter._procurecompare_original_init(self, *args, **kwargs)
    self._headers = getattr(self, "_headers", {})
    if not hasattr(self, "_session") and hasattr(self, "_client"):
        self._session = SimpleNamespace(headers=self._client._headers)

AuthenticatedOTLPExporter.__init__ = _compatible_exporter_init
agentops_processors.upload_logfile = lambda _trace_id: None

if agentops_api_key:
    os.environ["AGENTOPS_API_KEY"] = agentops_api_key
    agentops.init(
        api_key=agentops_api_key,
        auto_start_session=False,
        skip_auto_end_session=True,
        default_tags=["crewai"],
    )
    print("AgentOps enabled.")
else:
    print("AgentOps disabled (optional key not configured).")

print("API configuration loaded successfully.")

In [ ]:
from IPython.display import Markdown, display
from agentops.helpers.dashboard import get_trace_url

# Start one trace for the full procurement workflow and expose its replay link.
agentops_trace = None
if agentops_api_key:
    agentops_trace = agentops.start_trace(
        trace_name="crewai-procurement",
        tags=["crewai"],
    )
    if agentops_trace:
        replay_url = get_trace_url(agentops_trace.span)
        display(Markdown(f"🖇 **AgentOps — [Session Replay]({replay_url})**"))
else:
    print("AgentOps is disabled.")

In [ ]:
# CrewAI 0.114 strips the leading slash from absolute output paths.
# Run from the repository root and give CrewAI a safe relative path instead.
os.chdir(repository_root)
output_dir = "ai-agent-output"
os.makedirs(output_dir, exist_ok=True)
print(f"Workflow outputs: {(Path(output_dir).resolve())}")

tavily_api_key = get_secret("TAVILY_API_KEY")
scrapegraph_api_key = get_secret("SCRAPEGRAPH_API_KEY")
os.environ.setdefault("SGAI_TIMEOUT", "30")

# Use deterministic generation and initialize the two external data providers.
basic_llm = LLM(model="gpt-4o", temperature=0)
search_client = TavilyClient(api_key=tavily_api_key)
scrape_client = ScrapeGraphAI(api_key=scrapegraph_api_key)

In [ ]:
# Limit query generation and provide shared business context to every agent.
no_keywords = 10

about_company = "ProcureCompare AI helps organizations source products, compare prices, and make informed purchasing decisions."

company_context = StringKnowledgeSource(
    content=about_company
)

def get_agent_output(agent_label: str, task):
    """Return a native dict/list when possible so Jupyter renders it directly."""
    if task.output is None:
        raise RuntimeError(f"{agent_label} has no output. Run the workflow cell first.")
    try:
        return json.loads(task.output.raw)
    except (TypeError, json.JSONDecodeError):
        return task.output.raw

## Setup Agents

### Agent: A

In [ ]:
# Enforce a predictable JSON structure for the query-planning agent output.
class SuggestedSearchQueries(BaseModel):
    queries: List[str] = Field(..., title="Suggested search queries to be passed to the search engine",
                               min_length=1, max_length=no_keywords)

# Agent A converts the purchasing request into targeted e-commerce searches.
search_queries_recommendation_agent = Agent(
    role="Search Queries Recommendation Agent",
    goal="\n".join([
                "To provide a list of suggested search queries to be passed to the search engine.",
                "The queries must be varied and looking for specific items."
            ]),
    backstory="The agent is designed to help in looking for products by providing a list of suggested search queries to be passed to the search engine based on the context provided.",
    llm=basic_llm,
    verbose=True,
)

# The task writes its validated result to disk for inspection and reuse.
search_queries_recommendation_task = Task(
    description="\n".join([
        "The buyer is looking to purchase {product_name} using a value-for-money strategy.",
        "The campany target any of these websites to buy from: {websites_list}",
        "The company wants to reach all available proucts on the internet to be compared later in another stage.",
        "The stores must sell the product in {country_name}",
        "Generate at maximum {no_keywords} queries.",
        "The search keywords must be in {language} language.",
        "Search keywords must contains specific brands, types or technologies. Avoid general keywords.",
        "The search query must reach an ecommerce webpage for product, and not a blog or listing page."
    ]),
    expected_output="A JSON object containing a list of suggested search queries.",
    output_json=SuggestedSearchQueries,
    output_file=os.path.join(output_dir, "step_1_suggested_search_queries.json"),
    agent=search_queries_recommendation_agent
)

### Agent: B

In [ ]:
# Describe the normalized fields retained from every Tavily search result.
class SingleSearchResult(BaseModel):
    title: str
    url: str = Field(..., title="the page url")
    content: str
    score: float
    search_query: str

class AllSearchResults(BaseModel):
    results: List[SingleSearchResult]

# Keep an audit trail of URLs actually returned by Tavily.
verified_search_results = {}

# Expose a compact Tavily response so the agent does not invent URLs from truncated output.
@tool
def search_engine_tool(query: str):
    """Useful for search-based queries. Use this to find current information about any query related pages using a search engine"""
    response = search_client.search(
        query=query, search_depth="advanced", max_results=5,
        include_domains=["amazon.eg"], timeout=30,
    )
    results = []
    for item in response.get("results", []):
        url = item.get("url", "")
        score = float(item.get("score") or 0)
        if "amazon.eg" not in url.lower() or "/dp/" not in url.lower() or score < 0.10:
            continue
        record = {
            "title": item.get("title", ""), "url": url,
            "content": (item.get("content") or "")[:800],
            "score": score, "search_query": query,
        }
        verified_search_results[url] = record
        results.append(record)
    return {"query": query, "results": results}

# Agent B executes the proposed queries and filters irrelevant product pages.
search_engine_agent = Agent(
    role="Search Engine Agent",
    goal="To search for products based on the suggested search query",
    backstory="The agent is designed to help in looking for products by searching for products based on the suggested search queries.",
    llm=basic_llm,
    verbose=True,
    tools=[search_engine_tool]
)

def validate_search_output(_task_output):
    results = sorted(verified_search_results.values(), key=lambda item: item["score"], reverse=True)[:10]
    if len(results) < 3:
        return False, "Tavily returned fewer than 3 verified Amazon product pages."
    return True, AllSearchResults(results=results).model_dump_json(indent=2)

search_engine_task = Task(
    description="\n".join([
        "The task is to search for products based on the suggested search queries.",
        "You have to collect results from multiple search queries.",
        "Ignore any susbicious links or not an ecommerce single product website link.",
        "Ignore any search results with confidence score less than ({score_th}) .",
        "The search results will be used to compare prices of products from different websites.",
    ]),
    expected_output="A JSON object containing the search results.",
    output_file=os.path.join(output_dir, "step_2_search_results.json"),
    guardrail=validate_search_output,
    max_retries=0,
    agent=search_engine_agent
)

### Agent: C

In [ ]:
# Define the structured product data expected from each scraped page.
class ProductSpec(BaseModel):
    specification_name: str
    specification_value: str

class SingleExtractedProduct(BaseModel):
    page_url: str = Field(..., title="The original url of the product page")
    product_title: str = Field(..., title="The title of the product")
    product_image_url: str = Field(..., title="The url of the product image")
    product_url: str = Field(..., title="The url of the product")
    product_current_price: float = Field(..., title="The current price of the product")
    product_original_price: Optional[float] = Field(title="The original price of the product before discount. Set to None if no discount", default=None)
    product_discount_percentage: Optional[float] = Field(title="The discount percentage of the product. Set to None if no discount", default=None)

    product_specs: List[ProductSpec] = Field(..., title="The specifications of the product. Focus on the most important specs to compare.", min_length=1, max_length=5)

    agent_recommendation_rank: int = Field(..., title="The rank of the product to be considered in the final procurement report. (out of 5, Higher is Better) in the recommendation list ordering from the best to the worst")
    agent_recommendation_notes: List[str]  = Field(..., title="A set of notes why would you recommend or not recommend this product to the company, compared to other products.")


class AllExtractedProducts(BaseModel):
    products: List[SingleExtractedProduct]

successfully_scraped_products = {}

# Expose schema-guided ScrapeGraph extraction as a tool for Agent C.
@tool
def web_scraping_tool(page_url: str):
    """
    An AI Tool to help an agent to scrape a web page

    Example:
    web_scraping_tool(
        page_url="https://www.amazon.eg/example-product/dp/EXAMPLE"
    )
    """
    if page_url not in verified_search_results:
        return {"page_url": page_url, "details": None, "error": "Rejected: URL was not returned by Tavily."}

    try:
        result = scrape_client.extract(
            url=page_url,
            prompt="Extract the product details from this ecommerce product page.",
            schema=SingleExtractedProduct.model_json_schema(),
            fetch_config=FetchConfig(mode="fast", timeout=15000, country="EG"),
        )
        if result.status == "error":
            return {"page_url": page_url, "details": None, "error": result.error or "Extraction failed; try the next URL."}
        details = dict(result.data.json_data or {}) if result.data else None
        if details:
            details["page_url"] = page_url
            details["product_url"] = page_url
            details = SingleExtractedProduct.model_validate(details).model_dump()
            successfully_scraped_products[page_url] = details
    except Exception as error:
        return {"page_url": page_url, "details": None, "error": f"{error}; try the next URL."}

    return {
        "page_url": page_url,
        "details": details
    }

# Agent C visits the strongest search results and extracts comparable details.
scraping_agent = Agent(
    role="Web scraping agent",
    goal="To extract details from any website",
    backstory="The agent is designed to help in looking for required values from any website url. These details will be used to decide which best product to buy.",
    llm=basic_llm,
    tools=[web_scraping_tool],
    max_iter=6,
    max_execution_time=120,
    verbose=True,
)

def validate_scraping_output(_task_output):
    products = list(successfully_scraped_products.values())
    if not products:
        return False, "No verified product was extracted. Try another exact Tavily URL."
    return True, AllExtractedProducts(products=products).model_dump_json(indent=2)

scraping_task = Task(
    description="\n".join([
        "The task is to extract product details from any ecommerce store page url.",
        "The task has to collect results from multiple pages urls.",
        "Prefer amazon.eg product pages. If one URL times out, immediately try the next result.",
        "Collect the best {top_recommendations_no} products from the search results.",
    ]),
    expected_output="A JSON object containing products details",
    output_file=os.path.join(output_dir, "step_3_search_results.json"),
    guardrail=validate_scraping_output,
    max_retries=0,
    agent=scraping_agent
)

### Agent: D

In [ ]:
# Agent D turns the collected product data into the final human-readable report.
procurement_report_author_agent = Agent(
    role="Procurement Report Author Agent",
    goal="To generate a professional HTML page for the procurement report",
    backstory="The agent is designed to assist in generating a professional HTML page for the procurement report after looking into a list of products.",
    llm=basic_llm,
    verbose=True,
)

def clean_report_output(task_output):
    raw = task_output.raw.strip()
    start = raw.lower().find("<!doctype html")
    if start < 0:
        start = raw.lower().find("<html")
    end = raw.lower().rfind("</html>")
    if start < 0 or end < start:
        return False, "Return one complete raw HTML document."
    return True, raw[start:end + len("</html>")]

procurement_report_author_task = Task(
    description="\n".join([
        "The task is to generate a professional HTML page for the procurement report.",
        "You have to use Bootstrap CSS framework for a better UI.",
        "Use the provided context about the company to make a specialized report.",
        "The report will include the search results and prices of products from different websites.",
        "The report should be structured with the following sections:",
        "1. Executive Summary: A brief overview of the procurement process and key findings.",
        "2. Introduction: An introduction to the purpose and scope of the report.",
        "3. Methodology: A description of the methods used to gather and compare prices.",
        "4. Findings: Detailed comparison of prices from different websites, including tables and charts.",
        "5. Analysis: An analysis of the findings, highlighting any significant trends or observations.",
        "6. Recommendations: Suggestions for procurement based on the analysis.",
        "7. Conclusion: A summary of the report and final thoughts.",
        "8. Appendices: Any additional information, such as raw data or supplementary materials.",
        "Use only products and prices from the previous validated task. Never invent missing data.",
        "Return raw HTML only, beginning with <!DOCTYPE html>, without Markdown fences or prose.",
    ]),

    expected_output="A professional HTML page for the procurement report.",
    output_file=os.path.join(output_dir, "step_4_procurement_report.html"),
    guardrail=clean_report_output,
    max_retries=0,
    agent=procurement_report_author_agent,
)

## Run the AI Crew

In [ ]:
# Task order matters: each agent consumes the result produced by the previous one.
procure_compare_crew = Crew(
    agents=[
        search_queries_recommendation_agent,
        search_engine_agent,
        scraping_agent,
        procurement_report_author_agent,
    ],
    tasks=[
        search_queries_recommendation_task,
        search_engine_task,
        scraping_task,
        procurement_report_author_task,
    ],
    process=Process.sequential,
    knowledge_sources=[company_context]
)

In [ ]:
# Remove previous artifacts so a failed run cannot look successful.
for filename in ("step_1_suggested_search_queries.json", "step_2_search_results.json", "step_3_search_results.json", "step_4_procurement_report.html"):
    Path(output_dir, filename).unlink(missing_ok=True)

# Execute the workflow and always close the AgentOps trace with the correct status.
try:
    crew_results = procure_compare_crew.kickoff(
        inputs={
            "product_name": "coffee machine for the office",
            "websites_list": ["www.amazon.eg", "www.jumia.com.eg", "www.noon.com/egypt-en"],
            "country_name": "Egypt",
            "no_keywords": 10,
            "language": "English",
            "score_th": 0.10,
            "top_recommendations_no": 3,
        }
    )
except Exception:
    if agentops_trace:
        agentops.end_trace(agentops_trace, end_state="Error")
    raise
else:
    if agentops_trace:
        agentops.end_trace(agentops_trace, end_state="Success")

In [ ]:
agent_a_output = get_agent_output("Agent A — Query Planner", search_queries_recommendation_task)
agent_a_output

In [ ]:
agent_b_output = get_agent_output("Agent B — Search Engine", search_engine_task)
agent_b_output

In [ ]:
agent_c_output = get_agent_output("Agent C — Web Scraper", scraping_task)
agent_c_output

In [ ]:
agent_d_output = get_agent_output("Agent D — Report Author", procurement_report_author_task)
agent_d_output